# DS2 Core Notebook: Logistic Regression and the Odds Ratio

**Lesson:** DS2 · Logistic regression (Day 2, lesson 2) · **Pairs with:** ISLP Ch. 4, sections 4.1–4.3 · **Time:** ~25 min, in class

**Question:** which admitted patients are at high risk of dying in hospital?

**What you will do**
1. Heart failure: get the odds ratio from a 2×2 table, then the same OR from a logistic model
2. Age: an odds ratio for a continuous predictor, per year and per 10 years
3. Heart failure + age: crude vs adjusted OR
4. A full model with age, comorbidities, vitals and labs
5. A predicted risk for every patient, and the 10 highest-risk patients

**How to work:** run each cell in order with **Shift + Enter**. Cells marked **Your turn** need you to fill in a blank (`____`) or write a sentence.

In [ ]:
# Setup: load the CADets dataset (release 1)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

DATA_FILE = "cadets_admissions_release1_core.csv"
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/" + DATA_FILE

# Use a local copy if there is one; otherwise read the course copy from GitHub.
df = pd.read_csv(DATA_FILE if os.path.exists(DATA_FILE) else DATA_URL)
print(df.shape)

## Step 1 · A binary predictor: heart failure

### 1a. The 2×2 table (recap from EPI1)
`pd.crosstab` counts patients by heart failure (rows) and in-hospital death (columns).

In [ ]:
table = pd.crosstab(df["heart_failure"], df["in_hospital_death"])
table

### 1b. Odds and odds ratio by hand
Odds = died ÷ survived in each row. The odds ratio (OR) = odds with heart failure ÷ odds without.

In [ ]:
odds_hf = table.loc[1, 1] / table.loc[1, 0]      # heart failure = 1
odds_no_hf = table.loc[0, 1] / table.loc[0, 0]   # heart failure = 0
print("Odds with heart failure:   ", round(odds_hf, 3))
print("Odds without heart failure:", round(odds_no_hf, 3))
print("Odds ratio:                ", round(odds_hf / odds_no_hf, 2))

### 1c. The same question as a logistic regression
The formula `"in_hospital_death ~ heart_failure"` reads: *model the log-odds of in-hospital death using heart failure*. `.fit(disp=0)` fits the model quietly.

The model gives coefficients (β) on the **log-odds** scale. `np.exp(β)` turns them into **odds ratios**.

In [ ]:
model_hf = smf.logit("in_hospital_death ~ heart_failure", data=df).fit(disp=0)

print("Coefficients (log-odds scale):")
print(model_hf.params.round(3))
print()
print("exp(coefficients):")
print(np.exp(model_hf.params).round(3))

Look at the heart_failure row: **exp(β) is the same OR you got from the 2×2 table.** The intercept row, exp(β₀), is the odds of death for patients without heart failure.

A 95% confidence interval for the OR: exponentiate the interval of β.

In [ ]:
np.exp(model_hf.conf_int()).round(2)

### 1d. How did the model choose β = 1.024? The loss function
The model tries values of β and keeps the ones under which the outcomes we actually observed are **most likely**. That score is the **log-likelihood** (higher is better).

Flip its sign and average it over patients and you get the **log loss**, also called **cross-entropy**: the loss function of logistic regression (lower is better). The same loss trains tree ensembles (DS6) and neural networks (DS7).

In [ ]:
n = len(df)
print("Log-likelihood of the fitted model:", round(model_hf.llf, 1))
print("Log loss per patient (= -log-likelihood / n):", round(-model_hf.llf / n, 4))

# Compare with a model that ignores heart failure (intercept only)
model_null = smf.logit("in_hospital_death ~ 1", data=df).fit(disp=0)
print("Log loss with no predictor:", round(-model_null.llf / n, 4))

Adding heart failure lowers the log loss: the model explains the outcomes a little better. Any other value of β would give a higher log loss; Enrichment notebook A draws that curve.

### Your turn 1 · Say it in one sentence
Double-click this cell and complete the sentence using the OR and its 95% CI.

> Patients admitted with heart failure had ____ times the odds of dying in hospital compared with patients without heart failure (OR ____, 95% CI ____–____).

## Step 2 · A continuous predictor: age

There is no 2×2 table for age. The model still gives an OR, now **per one unit** of the predictor: one year of age.

In [ ]:
model_age = smf.logit("in_hospital_death ~ age", data=df).fit(disp=0)

beta_age = model_age.params["age"]
print("β for age:          ", round(beta_age, 4))
print("OR per 1 year:      ", round(np.exp(beta_age), 3))
print("OR per 10 years:    ", round(np.exp(10 * beta_age), 2))

To change the unit, multiply β **before** exponentiating: OR per 10 years = exp(10 × β). (Multiplying the OR by 10 is a common mistake.)

### 2b. Same OR, different change in risk
`model.predict` turns the model into a predicted risk. Here we ask for four ages.

In [ ]:
ages = pd.DataFrame({"age": [40, 50, 70, 80]})
ages["predicted_risk"] = model_age.predict(ages).round(3)
ages

From 40 to 50 and from 70 to 80 the odds are multiplied by the same 1.72, but the risk rises by about 1 percentage point in the first case and about 5 in the second. The OR is relative; the change in risk depends on where the patient starts.

### Your turn 2 · SpO₂
Fit a model with `spo2` only. Then compute the OR **per 1% lower** SpO₂ (hint: a lower value means multiplying β by −1).

In [ ]:
model_spo2 = smf.logit("in_hospital_death ~ ____", data=df).fit(disp=0)

beta_spo2 = model_spo2.params["spo2"]
print("OR per 1% higher SpO2:", round(np.exp(beta_spo2), 2))
print("OR per 1% lower SpO2: ", round(np.exp(____ * beta_spo2), 2))

Now complete the sentence (double-click to edit):

> Each 1% lower SpO₂ on admission was associated with ____ times the odds of in-hospital death.

## Step 3 · Crude vs adjusted: heart failure and age

Patients with heart failure are older, and age raises the risk of death on its own. Check the mean age in each group:

In [ ]:
df.groupby("heart_failure")["age"].mean().round(1)

Add age to the model with `+`. The heart failure OR is now **adjusted for age**: it compares patients with and without heart failure *at the same age*.

In [ ]:
model_hf_age = smf.logit("in_hospital_death ~ heart_failure + age", data=df).fit(disp=0)

comparison = pd.DataFrame({
    "crude OR": [np.exp(model_hf.params["heart_failure"]), np.exp(model_age.params["age"])],
    "adjusted OR": np.exp(model_hf_age.params[["heart_failure", "age"]]).values,
}, index=["heart_failure", "age"])
comparison.round(3)

## Step 4 · The full model

The same formula, with more terms. Each OR is read *holding all the other predictors constant*.

In [ ]:
formula = ("in_hospital_death ~ age + sex + heart_failure + diabetes + ckd + copd"
           " + active_cancer + heart_rate + systolic_bp + resp_rate + spo2"
           " + creatinine + hemoglobin + wbc")

model_full = smf.logit(formula, data=df).fit(disp=0)
print("Patients used:", int(model_full.nobs))

A small helper turns any fitted model into a readable table of ORs with 95% CIs and p-values. You can reuse it in later lessons.

In [ ]:
def or_table(model):
    """Odds ratios with 95% CI and p-values, without the intercept."""
    ci = np.exp(model.conf_int())
    out = pd.DataFrame({"OR": np.exp(model.params), "CI low": ci[0],
                        "CI high": ci[1], "p": model.pvalues})
    return out.drop(index="Intercept").round(3)

or_table(model_full).sort_values("OR", ascending=False)

### Your turn 3 · Read the table
1. What is the heart failure OR now? Compare it with the crude 2.78 and the age-adjusted 1.98.
2. Which predictors have a 95% CI that **crosses 1** (no clear association)?
3. CKD has an OR below 1. Does that mean CKD protects? (Hint: creatinine is also in the model.)

Write your answers here (double-click to edit):

## Step 5 · A predicted risk for every patient

`model.predict(df)` gives each admission a probability of in-hospital death between 0 and 1. This number is the product a ward team would see.

In [ ]:
df["predicted_risk"] = model_full.predict(df)
df["predicted_risk"].describe().round(3)

In [ ]:
plt.hist(df["predicted_risk"], bins=40, color="#0F6E7A")
plt.xlabel("Predicted risk of in-hospital death")
plt.ylabel("Number of patients")
plt.title("Most patients are low risk; a few are very high")
plt.show()

### 5b. The 10 highest-risk patients
Sort by predicted risk and show a few clinical columns next to it. The last column shows what actually happened.

In [ ]:
columns = ["patient_id", "age", "heart_failure", "spo2", "resp_rate",
           "creatinine", "predicted_risk", "in_hospital_death"]
top10 = df.sort_values("predicted_risk", ascending=False).head(10)[columns]
top10.round(2)

### Your turn 4 · A simple worklist rule
Suppose the ward reviews every patient with predicted risk above 30%. How many patients is that, and how many of them died?

In [ ]:
flagged = df[df["predicted_risk"] > ____]
print("Patients flagged:", len(flagged))
print("Of them died:    ", flagged["in_hospital_death"].sum())

Is that a good rule? How many deaths does it miss? That is exactly the question of the next lesson, **DS3 · Evaluation I** (thresholds, sensitivity, PPV, ROC curve).

## Key points
- Logistic regression models the **log-odds** of a yes/no outcome as a straight-line sum of predictors.
- **exp(β) is an odds ratio**: yes vs no for a binary predictor, per one unit for a continuous one (choose the unit: exp(10 × β) per 10 years).
- With one binary predictor the model reproduces the 2×2 table OR exactly.
- The model is fitted by maximizing the **log-likelihood**, which is the same as minimizing the **log loss** (cross-entropy).
- Adjusted ORs change with what else is in the model; always say what you adjusted for.
- `model.predict` gives every patient a predicted risk, the model’s real product.

**Further reading:** ISLP Ch. 4, sections 4.1–4.3. **Optional:** Enrichment notebook A (ISLP Lab 4.7, loss functions) and B (multinomial regression, GLMs for length of stay).